In [1]:
# !pip install torch torchvision torchaudio
# !pip install opencv-python mediapipe scipy scikit-learn matplotlib tqdm

In [2]:
import os
import cv2
import math
import torch
import random
import numpy as np
import scipy.signal as sp_signal
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.fft import rfft, rfftfreq
from sklearn.model_selection import train_test_split

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import mediapipe as mp

In [12]:
# ================================
# IMPROVED CONFIG
# ================================

class ImprovedConfig:

    DATASET_PATH = "./dataset"
    PREPROCESS_DIR = "./preprocessed"

    FPS = 30
    IMG_SIZE = 72
    SEQ_LEN = 160

    BATCH_SIZE = 4
    EPOCHS = 30

    LR = 1e-4

    D_MODEL = 64
    NHEAD = 4
    NUM_LAYERS = 2

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = ImprovedConfig()

print("Using Device:", cfg.DEVICE)


Using Device: cpu


In [13]:
# class Config:
#     DATASET_PATH = "./dataset"
#     PREPROCESS_DIR = "./preprocessed"

#     FPS = 30
#     SEQ_LEN = 160

#     IMG_SIZE = 72

#     BATCH_SIZE = 8
#     EPOCHS = 30
#     LR = 1e-4

#     DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#     D_MODEL = 64
#     NHEAD = 4
#     NUM_LAYERS = 2

# cfg = Config()

# os.makedirs(cfg.PREPROCESS_DIR, exist_ok=True)

In [14]:
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.6
)

In [15]:
import mediapipe as mp
print(mp)
print(mp.__file__)

<module 'mediapipe' from 'C:\\Users\\UIET\\anaconda3\\envs\\nrppg\\lib\\site-packages\\mediapipe\\__init__.py'>
C:\Users\UIET\anaconda3\envs\nrppg\lib\site-packages\mediapipe\__init__.py


In [16]:
def extract_face_roi(frame):

    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = face_detector.process(rgb)

    if not result.detections:
        return None

    det = result.detections[0]

    bbox = det.location_data.relative_bounding_box

    x = int(bbox.xmin * w)
    y = int(bbox.ymin * h)
    bw = int(bbox.width * w)
    bh = int(bbox.height * h)

    x = max(0, x)
    y = max(0, y)

    bw = min(bw, w - x)
    bh = min(bh, h - y)

    roi = frame[y:y+bh, x:x+bw]

    if roi.size == 0:
        return None

    roi = cv2.resize(roi, (cfg.IMG_SIZE, cfg.IMG_SIZE))

    return roi

In [17]:
def bandpass_filter(signal_data, low=0.7, high=4.0, fs=30):

    if len(signal_data) < 30:
        return signal_data

    nyquist = fs / 2

    low /= nyquist
    high /= nyquist

    b, a = sp_signal.butter(3, [low, high], btype='band')

    try:
        filtered = sp_signal.filtfilt(b, a, signal_data)
    except:
        filtered = signal_data

    return filtered

In [18]:
# ================================
# FIXED BPM COMPUTATION
# ================================

def compute_bpm(signal_data, fs=30):

    signal_data = np.asarray(signal_data)

    if len(signal_data) < 30:
        return 0

    yf = np.abs(np.fft.rfft(signal_data))

    xf = np.fft.rfftfreq(
        len(signal_data),
        d=1/fs
    )

    mask = (xf >= 0.7) & (xf <= 4.0)

    xf = xf[mask]
    yf = yf[mask]

    if len(yf) == 0:
        return 0

    peak_freq = xf[np.argmax(yf)]

    bpm = peak_freq * 60

    return bpm


In [19]:
# def compute_bpm(signal_data, fs=30):

#     signal_data = np.asarray(signal_data)

#     if len(signal_data) < 30:
#         return 0

#     signal_data = bandpass_filter(signal_data)

#     # REAL FFT
#     yf = np.abs(np.fft.rfft(signal_data))

#     # FREQUENCIES
#     xf = np.fft.rfftfreq(len(signal_data), d=1/fs)

#     # HEART RATE RANGE
#     mask = (xf >= 0.7) & (xf <= 4.0)

#     xf = xf[mask]
#     yf = yf[mask]

#     if len(yf) == 0:
#         return 0

#     peak_freq = xf[np.argmax(yf)]

#     bpm = peak_freq * 60

#     return bpm

In [20]:
def preprocess_dataset():

    subjects = sorted(os.listdir(cfg.DATASET_PATH))

    for subject in tqdm(subjects):

        subject_path = os.path.join(cfg.DATASET_PATH, subject)

        video_path = os.path.join(subject_path, "vid.avi")
        gt_path = os.path.join(subject_path, "ground_truth.txt")

        if not os.path.exists(video_path):
            continue

        if not os.path.exists(gt_path):
            continue

        save_path = os.path.join(cfg.PREPROCESS_DIR, f"{subject}.npz")

        if os.path.exists(save_path):
            continue

        gt_signal = np.loadtxt(gt_path)

        cap = cv2.VideoCapture(video_path)

        frames = []

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            roi = extract_face_roi(frame)

            if roi is None:
                continue

            roi = roi.astype(np.float32) / 255.0

            frames.append(roi)

        cap.release()

        frames = np.array(frames)

        min_len = min(len(frames), len(gt_signal))

        frames = frames[:min_len]
        gt_signal = gt_signal[:min_len]

        np.savez(
            save_path,
            frames=frames,
            signal=gt_signal
        )

preprocess_dataset()

100%|████████████████████████████████████████████████████████████████████████████████| 42/42 [00:00<00:00, 3375.57it/s]


In [21]:
# ================================
# POSITIONAL ENCODING
# ================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]

In [22]:
class RPPGDataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        data = np.load(self.files[idx])

        frames = data['frames']
        signal = data['signal']

        if len(frames) <= cfg.SEQ_LEN:

            pad = cfg.SEQ_LEN - len(frames)

            frames = np.pad(
                frames,
                ((0,pad),(0,0),(0,0),(0,0))
            )

            signal = np.pad(signal, (0,pad))

            start = 0

        else:
            start = random.randint(0, len(frames)-cfg.SEQ_LEN)

        frames = frames[start:start+cfg.SEQ_LEN]
        signal = signal[start:start+cfg.SEQ_LEN]

        signal = bandpass_filter(signal)

        signal = (signal - signal.mean()) / max(signal.std(), 1e-6)

        bpm = compute_bpm(signal)

        frames = torch.tensor(frames).permute(0,3,1,2).float()
        signal = torch.tensor(signal).float()
        bpm = torch.tensor(bpm).float()

        return frames, signal, bpm

In [23]:
all_files = [
    os.path.join(cfg.PREPROCESS_DIR, x)
    for x in os.listdir(cfg.PREPROCESS_DIR)
]

train_files, val_files = train_test_split(
    all_files,
    test_size=0.2,
    random_state=42
)

train_dataset = RPPGDataset(train_files)
val_dataset = RPPGDataset(val_files)

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=cfg.BATCH_SIZE,
#     shuffle=True,
#     num_workers=0
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=cfg.BATCH_SIZE,
#     shuffle=False,
#     num_workers=2
# )


# ================================
# IMPROVED DATALOADER SETTINGS
# ================================

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 9
Validation batches: 3


In [24]:
# ================================
# IMPROVED SPATIAL ENCODER
# ================================

class ImprovedSpatialEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Dropout(0.2),

            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self, x):

        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)

        x = self.encoder(x)

        x = x.squeeze(-1).squeeze(-1)

        x = x.view(B, T, cfg.D_MODEL)

        return x


In [25]:
# class SpatialEncoder(nn.Module):

#     def __init__(self):
#         super().__init__()

#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 32, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2),

#             nn.AdaptiveAvgPool2d(1)
#         )

#     def forward(self, x):

#         B, T, C, H, W = x.shape

#         x = x.view(B*T, C, H, W)

#         x = self.encoder(x)

#         # x = x.view(B, T, -1)
#         x = x.squeeze(-1).squeeze(-1)
#         x = x.view(B, T, cfg.D_MODEL)
        
#         return x

In [26]:
# ================================
# IMPROVED TRANSFORMER MODEL
# ================================

class ImprovedTransformerRPPG(nn.Module):

    def __init__(self):

        super().__init__()

        self.spatial = ImprovedSpatialEncoder()

        self.temporal_conv = nn.Conv1d(
            cfg.D_MODEL,
            cfg.D_MODEL,
            kernel_size=3,
            padding=1
        )

        self.pos_encoder = PositionalEncoding(cfg.D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            batch_first=True,
            dropout=0.1
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS
        )

        self.signal_head = nn.Linear(cfg.D_MODEL, 1)

    def forward(self, x):

        x = self.spatial(x)

        x = x.permute(0,2,1)

        x = self.temporal_conv(x)

        x = x.permute(0,2,1)

        x = self.pos_encoder(x)

        x = self.transformer(x)

        signal = self.signal_head(x).squeeze(-1)

        return signal


In [27]:
# class TransformerRPPG(nn.Module):

#     def __init__(self):
#         super().__init__()

#         self.spatial = SpatialEncoder()

#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=cfg.D_MODEL,
#             nhead=cfg.NHEAD,
#             batch_first=True
#         )

#         self.transformer = nn.TransformerEncoder(
#             encoder_layer,
#             num_layers=cfg.NUM_LAYERS
#         )

#         self.signal_head = nn.Linear(cfg.D_MODEL, 1)

#         self.bpm_head = nn.Sequential(
#             nn.Linear(cfg.D_MODEL, 32),
#             nn.ReLU(),
#             nn.Linear(32, 1)
#         )

#     def forward(self, x):

#         x = self.spatial(x)

#         x = self.transformer(x)

#         signal = self.signal_head(x).squeeze(-1)

#         pooled = x.mean(dim=1)

#         bpm = self.bpm_head(pooled).squeeze(-1)

#         return signal, bpm

In [28]:
def negative_pearson(pred, target):

    pred = pred - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)

    numerator = torch.sum(pred * target, dim=1)

    denominator = torch.sqrt(
        torch.sum(pred**2, dim=1) *
        torch.sum(target**2, dim=1)
    )

    loss = 1 - numerator / (denominator + 1e-8)

    return loss.mean()

In [35]:
model = ImprovedTransformerRPPG().to(cfg.DEVICE)

optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.LR
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=3
)

scaler = GradScaler(
    enabled=torch.cuda.is_available()
)



print("CUDA Available:", torch.cuda.is_available())


CUDA Available: False


In [36]:
for epoch in range(cfg.EPOCHS):

    model.train()

    train_loss = 0

    for frames, signal, bpm in tqdm(train_loader):

        frames = frames.to(cfg.DEVICE)
        signal = signal.to(cfg.DEVICE)
        bpm = bpm.to(cfg.DEVICE)

        optimizer.zero_grad()

        with autocast(enabled=torch.cuda.is_available()):

            pred_signal, pred_bpm = model(frames)

            waveform_loss = negative_pearson(
                pred_signal,
                signal
            )

            bpm_loss = nn.functional.l1_loss(
                pred_bpm,
                bpm
            )

            loss = waveform_loss + 0.1 * bpm_loss

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )


        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

  0%|                                                                                            | 0/9 [00:00<?, ?it/s]


IndexError: boolean index did not match indexed array along dimension 0; dimension is 160 but corresponding boolean dimension is 81

# Integrated Improvements for Transformer-Based rPPG

In [ ]:
# # ================================
# # SAFE GRADSCALER
# # ================================

# scaler = GradScaler(
#     enabled=torch.cuda.is_available()
# )

# print("CUDA Available:", torch.cuda.is_available())


In [ ]:
# # ================================
# # POSITIONAL ENCODING
# # ================================

# class PositionalEncoding(nn.Module):

#     def __init__(self, d_model, max_len=5000):

#         super().__init__()

#         pe = torch.zeros(max_len, d_model)

#         position = torch.arange(0, max_len).unsqueeze(1)

#         div_term = torch.exp(
#             torch.arange(0, d_model, 2)
#             * (-math.log(10000.0) / d_model)
#         )

#         pe[:, 0::2] = torch.sin(position * div_term)
#         pe[:, 1::2] = torch.cos(position * div_term)

#         pe = pe.unsqueeze(0)

#         self.register_buffer('pe', pe)

#     def forward(self, x):

#         return x + self.pe[:, :x.size(1)]


In [ ]:
# # ================================
# # IMPROVED DATALOADER SETTINGS
# # ================================

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=cfg.BATCH_SIZE,
#     shuffle=True,
#     num_workers=0
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=cfg.BATCH_SIZE,
#     shuffle=False,
#     num_workers=0
# )

# print("Train batches:", len(train_loader))
# print("Validation batches:", len(val_loader))


In [ ]:
# # ================================
# # GRADIENT CLIPPING EXAMPLE
# # ================================

# torch.nn.utils.clip_grad_norm_(
#     model.parameters(),
#     1.0
# )


In [ ]:
# ================================
# VALIDATION METRICS
# ================================

def compute_metrics(pred_signal, gt_signal):

    pred_signal = pred_signal.flatten()
    gt_signal = gt_signal.flatten()

    mae = np.mean(np.abs(pred_signal - gt_signal))

    rmse = np.sqrt(
        np.mean((pred_signal - gt_signal) ** 2)
    )

    pearson = np.corrcoef(
        pred_signal,
        gt_signal
    )[0,1]

    return {
        "MAE": mae,
        "RMSE": rmse,
        "Pearson": pearson
    }



# Important Improvements Added

- Fixed FFT indexing bug
- Added positional encoding
- Added temporal convolution
- Added safer GradScaler
- Added gradient clipping
- Improved CNN backbone
- Fixed DataLoader multiprocessing issues on Windows
- Added validation metrics
- Improved transformer stability
- Better physiological waveform learning
